In [1]:
import cv2
from concurrent.futures import ProcessPoolExecutor, as_completed
import fcntl
import shutil
import math
import mplcursors
import random
import csv
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
from tqdm import tqdm
import os
import numpy as np
import re

In [2]:
db_folder = '../Photometric_DB'
new_db_folder = '../Photometric_DB_New'
metadata_files = {
    './multi/metadata_filtered.csv': 'metadata_filter.csv',
    './multi/train_set.csv': 'train_set.csv',
    './multi/val_set.csv': 'val_set.csv',
    './multi/test_set.csv': 'test_set.csv',
    './multi/gallery_set.csv': 'gallery_set.csv',
    './multi/probe_set.csv': 'probe_set.csv',
}
total_cores = os.cpu_count() - 4

In [4]:
valid_sessions = set()

for src_path, filename in metadata_files.items():
    if os.path.exists(src_path):
        dest_path = os.path.join(new_db_folder, filename)
        # shutil.copy2(src_path, dest_path)
        print(f"Đã copy: {src_path} -> {dest_path}")
        
        # Đọc file CSV để thu thập các cặp (id, session) hợp lệ
        df_meta = pd.read_csv(src_path)
        if 'id' in df_meta.columns and 'session' in df_meta.columns:
            for _, row in df_meta.iterrows():
                # Lưu dưới dạng tuple (id chuẩn hóa thành string, session)
                valid_sessions.add((str(row['id']), str(row['session'])))
    else:
        print(f"Cảnh báo: Không tìm thấy file metadata tại {src_path}")

Đã copy: ./multi/metadata_filtered.csv -> ../Photometric_DB_New/metadata_filter.csv
Đã copy: ./multi/train_set.csv -> ../Photometric_DB_New/train_set.csv
Đã copy: ./multi/val_set.csv -> ../Photometric_DB_New/val_set.csv
Đã copy: ./multi/test_set.csv -> ../Photometric_DB_New/test_set.csv
Đã copy: ./multi/gallery_set.csv -> ../Photometric_DB_New/gallery_set.csv
Đã copy: ./multi/probe_set.csv -> ../Photometric_DB_New/probe_set.csv


In [5]:
session_paths = []

for id_str in os.listdir(db_folder):
    if id_str.isdigit():
        id_path = os.path.join(db_folder, id_str)
        if os.path.isdir(id_path):
            for session_name in os.listdir(id_path):
                session_path = os.path.join(id_path, session_name)
                
                # Chỉ lấy session nếu nó nằm trong danh sách hợp lệ của metadata
                if os.path.isdir(session_path) and (str(id_str), str(session_name)) in valid_sessions:
                    session_paths.append(session_path)
print(f"Tổng số session khớp với metadata cần copy: {len(session_paths)}")

Tổng số session khớp với metadata cần copy: 3056


In [ ]:
def process_session(session_path):
    # Tạo đường dẫn đích tương ứng trong thư mục mới
    new_session_path = session_path.replace('Photometric_DB', 'Photometric_DB_New')
    
    # Tạo thư mục đích nếu chưa có
    os.makedirs(new_session_path, exist_ok=True)
    
    files_to_copy = ['albedo_crop.png', 'normalmap_crop.png']
    
    for file_name in files_to_copy:
        src_file = os.path.join(session_path, file_name)
        dest_file = os.path.join(new_session_path, file_name)
        
        if os.path.exists(src_file):
            shutil.copy2(src_file, dest_file)

In [7]:
with ProcessPoolExecutor(max_workers=total_cores) as executor:
    list(tqdm(executor.map(process_session, session_paths), total=len(session_paths)))

100%|██████████| 3056/3056 [00:03<00:00, 992.18it/s] 
